# Django Caching & Redis

## Why Caching?

Without caching, every request triggers a database query. If 1,000 users request the same book list in one minute, the database runs 1,000 identical queries.

Caching stores the result for a configured time (TTL). The database runs once; subsequent requests get the cached result.

**Benefits:**
- Faster response times
- Fewer database queries
- Handles more traffic with the same resources


## Cache Backends

| Backend | Storage | Notes |
|---------|---------|-------|
| `LocMemCache` | Python process memory | Fastest; not shared across processes |
| `FileBasedCache` | Filesystem | Persists across restarts; slower |
| `DatabaseCache` | DB table | Shared; slower |
| `Memcached` | In-memory | High-performance; external service |
| **Redis** | In-memory + persistence | Recommended; supports clustering |


## LocMemCache Setup and @cache_page

```python
# settings.py
CACHES = {
    "default": {
        "BACKEND": "django.core.cache.backends.locmem.LocMemCache",
        "LOCATION": "unique",
        "TIMEOUT": 60,
        "KEY_PREFIX": "myapp",
    }
}
```

Cache an entire view response:
```python
from django.utils.decorators import method_decorator
from django.views.decorators.cache import cache_page
from rest_framework import viewsets

CACHE_TTL = 60

class BookViewSet(viewsets.ModelViewSet):
    @method_decorator(cache_page(CACHE_TTL, key_prefix='books_list'))
    def list(self, request, *args, **kwargs):
        return super().list(request, *args, **kwargs)
```


## Low-Level Cache API

For fine-grained control, use the cache API directly:

```python
from django.core.cache import cache
from rest_framework.decorators import api_view
from rest_framework.response import Response

@api_view(['GET'])
def bestsellers(request):
    data = cache.get('bestsellers:v1')
    if not data:
        top = list(Book.objects.order_by('-sales_count').values(
            'id', 'title', 'author', 'price', 'sales_count'
        )[:10])
        data = {'count': len(top), 'results': top}
        cache.set('bestsellers:v1', data, timeout=60)
    return Response(data)
```

Key API methods:
- `cache.set(key, value, timeout)` — store a value
- `cache.get(key)` — retrieve a value (returns `None` if missing)
- `cache.delete(key)` — remove a value


## Cache Invalidation with Signals

When data changes, stale cache entries must be deleted (invalidated):

```python
# store/signals.py
from django.core.cache import cache
from django.db.models.signals import post_save, post_delete
from django.dispatch import receiver
from .models import Book

def invalidate_book_caches():
    cache.delete('bestsellers:v1')

@receiver(post_save, sender=Book)
def on_book_saved(sender, instance, created, **kwargs):
    invalidate_book_caches()

@receiver(post_delete, sender=Book)
def on_book_deleted(sender, instance, **kwargs):
    invalidate_book_caches()
```

Register signals in `apps.py`:
```python
def ready(self):
    import store.signals
```


## Redis Cache Backend

```bash
pip install django-redis redis
redis-server        # start Redis locally
redis-cli ping      # should return PONG
```

```python
# settings.py
CACHES = {
    "default": {
        "BACKEND": "django_redis.cache.RedisCache",
        "LOCATION": "redis://127.0.0.1:6379/1",
        "OPTIONS": {
            "CLIENT_CLASS": "django_redis.client.DefaultClient",
            "IGNORE_EXCEPTIONS": True,
        },
        "TIMEOUT": 60,
        "KEY_PREFIX": "myapp",
    }
}
```

Inspect cached keys:
```bash
redis-cli -n 1
KEYS *
TTL "myapp:bestsellers:v1"
```


## Choosing a Cache TTL

The right TTL depends on how often data changes and how expensive it is to recompute:

- **Frequently changing data** → short TTL (e.g. 10–30 seconds)
- **Rarely changing, expensive data** → long TTL (e.g. minutes to hours)

Always invalidate the cache proactively when the underlying data changes, rather than relying solely on TTL expiry.

A cached value should be safe to lose: if Redis is restarted or a key expires, the application must be able to regenerate the data from the database.


## Summary

- Caching reduces database load by storing computed results temporarily.
- `LocMemCache` is fast but not shared across processes; Redis is preferred in production.
- `@cache_page` caches an entire view response; the low-level API caches arbitrary values.
- Always invalidate caches with signals when the underlying data changes.
- Set `KEY_PREFIX` to avoid collisions when multiple apps share the same Redis instance.
- A cached value should always be regenerable from the database — caching is about performance, not persistence.
